In [1]:
pip install gradio PyGithub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.3 MB/s eta 0:00:00


In [23]:
from github import Github
import github
import base64
import os

def upload_file_to_github(repo_name, file_name, file_content_bytes, github_token, branch="main", commit_message="Add file via Gradio app", folder_path=""):
    try:
        g = Github(github_token)
        user = g.get_user()
        username = user.login
        try:
            repo = g.get_repo(f"{username}/{repo_name}")
        except github.UnknownObjectException:
            try:
                repo = user.create_repo(repo_name)
            except Exception as e:
                return f"Error creating repository: {e}"
        full_path = os.path.join(folder_path, file_name).strip('/')
        try:
            contents = repo.get_contents(full_path, ref=branch)
            repo.update_file(contents.path, commit_message, file_content_bytes, contents.sha, branch=branch)
            return f"Successfully updated '{full_path}' in '{repo_name}' on branch '{branch}'."
        except (github.UnknownObjectException, github.GithubException) as ge:
            error_msg = str(ge).lower()
            if isinstance(ge, github.UnknownObjectException) or "no commit found" in error_msg or "404" in error_msg:
                try:
                    repo.create_file(full_path, commit_message, file_content_bytes, branch=branch)
                    return f"Successfully created '{full_path}' in '{repo_name}' on branch '{branch}'."
                except github.GithubException as e2:
                    if "no commit found" in str(e2).lower():
                        repo.create_file(full_path, commit_message, file_content_bytes)
                        return f"Initialized repository with '{full_path}' as the first file."
                    return f"GitHub Error during creation: {e2}"
            return f"GitHub Error: {ge}"
        except Exception as e:
            return f"File operation error: {e}"
    except Exception as e:
        return f"Authentication Error: {e}"

def delete_file_from_github(repo_name, file_path, github_token, branch="main", commit_message="Delete file via Gradio app"):
    try:
        g = Github(github_token)
        user = g.get_user()
        repo = g.get_repo(f"{user.login}/{repo_name}")
        try:
            contents = repo.get_contents(file_path, ref=branch)
            repo.delete_file(contents.path, commit_message, contents.sha, branch=branch)
            return f"Successfully deleted '{file_path}' from '{repo_name}' on branch '{branch}'."
        except github.UnknownObjectException:
            return f"Error: The file '{file_path}' was not found. Please check the path."
    except Exception as e:
        return f"Deletion error: {e}"

def list_repo_contents(repo_name, github_token, branch="main"):
    if not repo_name or not github_token:
        return gr.Dropdown(choices=[])
    try:
        g = Github(github_token)
        user = g.get_user()
        repo = g.get_repo(f"{user.login}/{repo_name}")
        contents = repo.get_contents("", ref=branch)
        files = []
        while contents:
            file_content = contents.pop(0)
            if file_content.type == "dir":
                contents.extend(repo.get_contents(file_content.path, ref=branch))
            else:
                files.append(file_content.path)
        return gr.Dropdown(choices=sorted(files), label="Select File to Delete")
    except Exception as e:
        return gr.Dropdown(choices=[f"Error: {str(e)}"])

def get_user_repos(token):
    if not token:
        return gr.Dropdown(choices=[])
    try:
        g = Github(token)
        repos = [repo.name for repo in g.get_user().get_repos()]
        return gr.Dropdown(choices=sorted(repos), label="2. Target Repository")
    except Exception as e:
        return gr.Dropdown(choices=[f"Error: {str(e)}"])

print("Logic updated with browsing support.")

Logic updated with browsing support.


In [24]:
import gradio as gr
import os

def github_uploader_interface(github_repo_name, github_token, branch_name, folder_path, uploaded_file, commit_message):
    if not uploaded_file:
        return "Please upload a file."
    if not github_repo_name or not github_token:
        return "Repo name and Token are required."
    file_name = uploaded_file.name.split('/')[-1]
    with open(uploaded_file.name, 'rb') as f:
        file_data = f.read()
    return upload_file_to_github(github_repo_name, file_name, file_data, github_token, branch_name or "main", commit_message or "Add file via Gradio app", folder_path)

def github_deleter_interface(github_repo_name, github_token, branch_name, file_path_selection, commit_message):
    if not github_repo_name or not github_token or not file_path_selection:
        return "Repo name, Token, and File Selection are required for deletion."
    return delete_file_from_github(github_repo_name, file_path_selection, github_token, branch_name or "main", commit_message or "Delete file via Gradio app")

with gr.Blocks() as demo:
    gr.Markdown("# Advanced GitHub File Manager (Upload & Delete)")
    with gr.Row():
        token_input = gr.Textbox(label="1. GitHub PAT", type="password", placeholder="Enter your token first")
        fetch_repos_btn = gr.Button("Fetch My Repositories")
    repo_input = gr.Dropdown(label="2. Target Repository", allow_custom_value=True)
    with gr.Row():
        branch_input = gr.Textbox(label="Branch", value="main")
        folder_input = gr.Textbox(label="Upload Subfolder (Optional)", placeholder="e.g. images")
    with gr.Tab("Upload"):
        file_input = gr.File(label="Select File to Upload")
        upload_commit = gr.Textbox(label="Commit Message (Upload)", placeholder="Optional")
        upload_btn = gr.Button("Upload to GitHub", variant="primary")
    with gr.Tab("Browse & Delete"):
        fetch_files_btn = gr.Button("Fetch Repository Files")
        file_path_del = gr.Dropdown(label="Select File to Delete", allow_custom_value=True)
        delete_commit = gr.Textbox(label="Commit Message (Delete)", placeholder="Optional")
        delete_btn = gr.Button("Delete from GitHub", variant="stop")
    output = gr.Textbox(label="Status Output")
    fetch_repos_btn.click(fn=get_user_repos, inputs=token_input, outputs=repo_input)
    fetch_files_btn.click(fn=list_repo_contents, inputs=[repo_input, token_input, branch_input], outputs=file_path_del)
    upload_btn.click(fn=github_uploader_interface, inputs=[repo_input, token_input, branch_input, folder_input, file_input, upload_commit], outputs=output)
    delete_btn.click(fn=github_deleter_interface, inputs=[repo_input, token_input, branch_input, file_path_del, delete_commit], outputs=output)

demo.launch(share=True, inline=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2de5bc461ebbc1eee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
import pandas as pd
import os

print("Required libraries are imported successfully!!")

Required libraries are imported successfully!!
